# 11 - Dış Doğrulama

Bu notebook, sabit finalist modelleri Dataset3 external test spliti üzerinde classification ve detection açısından değerlendirir.

Amaç yeni model eğitmek veya Dataset3 üzerinde parametre seçmek değil; önceki aşamalarda belirlenen finalistlerin dış veri üzerindeki davranışını ölçmektir.


## 1. Kütüphaneler

Bu bölümde tablo işlemleri, görüntü işleme, model yükleme, feature çıkarma ve metrik hesaplama için kullanılan kütüphaneler yüklenir.


In [ ]:
from pathlib import Path
from collections import OrderedDict
from datetime import datetime
import gc, math, os, random, time, warnings

import numpy as np
import pandas as pd

import cv2
import joblib
from skimage.feature import hog, local_binary_pattern

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("once")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

print("Kütüphaneler yüklendi.")


## 2. Ayarlar

Bu bölümde smoke/full çalışma modu, dış doğrulama adımları, overwrite davranışı ve feature ayarları tanımlanır.


In [ ]:
STAGE_NAME = "11_external_validation"
NOTEBOOK_NAME = "01_external_validation_classification_and_detection"
DATASET3_NAME = "dataset3_external"
DATASET3_POSITIVE_CLASS_IDS = {0}

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Run smoke first. After smoke passes, manually set False and rerun for the official full run.
RUN_SMOKE_TEST = False

RUN_EXPLORATION_AUDIT = True
RUN_EXTERNAL_CLASSIFICATION_METADATA = True
RUN_EXTERNAL_CLASSIFICATION_FEATURES = True
RUN_EXTERNAL_CLASSIFICATION_EVALUATION = True
RUN_EXTERNAL_DETECTION = True
COMPUTE_AP_MAP = True

OVERWRITE_EXTERNAL_PATCH_METADATA = False
OVERWRITE_EXTERNAL_FEATURES = False
OVERWRITE_PARTIAL_DETECTION_RESULTS = False
OVERWRITE_TABLES = False

SMOKE_TEST_IMAGE_LIMIT = 5
SMOKE_TEST_MAX_POSITIVE_PATCHES = 5

FEATURE_EXTRACTION_BATCH_SIZE = 512
MAX_CANDIDATES_PER_IMAGE = 2000
IMAGE_CACHE_MAX_ITEMS = 64

MATCHING_IOU_THRESHOLDS = [0.20, 0.30]
AP_IOU_THRESHOLDS = [0.20, 0.30, 0.50]
MAP_50_95_THRESHOLDS = [round(x, 2) for x in np.arange(0.50, 0.96, 0.05)]

# Stage 03 feature settings
HOG_ORIENTATIONS = 9
HOG_CELLS_PER_BLOCK = (2, 2)
HOG_BLOCK_NORM = "L2-Hys"
HOG_TRANSFORM_SQRT = True

HSV_H_BINS = 16
HSV_S_BINS = 8
HSV_V_BINS = 8

LBP_RADIUS = 1
LBP_POINTS = 8 * LBP_RADIUS
LBP_METHOD = "uniform"

HOG_CONFIGS = {
    "hog8": {"pixels_per_cell": (8, 8)},
    "hog16": {"pixels_per_cell": (16, 16)},
}

FEATURE_SET_CONFIG = {
    "hog16_pca_hsv_lbp": {"hog_variant": "hog16", "use_pca": True, "include_hsv": True, "include_lbp": True},
    "hog16_hsv_lbp": {"hog_variant": "hog16", "use_pca": False, "include_hsv": True, "include_lbp": True},
}

EXTERNAL_PATCH_SET_CONFIGS = {
    "centered80_balanced": {"patch_size": 80, "ratio_variant": "balanced", "negative_multiplier": 1},
    "centered80_neg4x": {"patch_size": 80, "ratio_variant": "neg4x", "negative_multiplier": 4},
    "centered48_neg4x": {"patch_size": 48, "ratio_variant": "neg4x", "negative_multiplier": 4},
}

NEGATIVE_TYPES = ["center_negative", "mixed_negative", "outer_negative"]

# Deterministic patch-set-specific offsets. Do not use Python hash() here because it is randomized across sessions.
PATCH_SET_SEED_OFFSETS = {
    "centered80_balanced": 101,
    "centered80_neg4x": 102,
    "centered48_neg4x": 103,
}
MAX_NEGATIVE_SAMPLING_ATTEMPTS = 500
NEGATIVE_MAX_IOU_WITH_ANY_GT = 0.05

NOTEBOOK_STARTED_AT = datetime.now()
print(f"[START] {NOTEBOOK_NAME} | {NOTEBOOK_STARTED_AT:%Y-%m-%d %H:%M:%S}")
print(f"[INFO] RUN_SMOKE_TEST={RUN_SMOKE_TEST}")
print("Ayarlar yüklendi.")


## 3. Dosya Yolları ve Kayıt Yardımcıları

Bu bölümde Dataset3 test klasörleri, Stage 11 çıktı klasörü, external feature klasörleri ve kayıt yardımcıları tanımlanır.


In [ ]:
def find_project_root(start_path=None):
    start = Path(start_path or Path.cwd()).resolve()
    for candidate in [start] + list(start.parents):
        has_repo_layout = (candidate / "approaches" / "traditional_ml").exists()
        has_data_dir = (candidate / "data").exists()
        if has_repo_layout and has_data_dir:
            return candidate
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Proje kökü bulunamadı. Notebook'u repo içinde çalıştırın.")

APPROACH_ROOT = PROJECT_ROOT / "approaches" / "traditional_ml"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATASET3_ROOT = RAW_DATA_DIR / DATASET3_NAME
DATASET3_TEST_IMAGES_DIR = DATASET3_ROOT / "test" / "images"
DATASET3_TEST_LABELS_DIR = DATASET3_ROOT / "test" / "labels"

RUN_MODE = "smoke" if RUN_SMOKE_TEST else "full"
OUTPUT_DIR = APPROACH_ROOT / "outputs" / STAGE_NAME / ("01_external_validation_smoke" if RUN_SMOKE_TEST else "01_external_validation")
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
PARTIAL_DIR = TABLES_DIR / "partial_model_results"
INPUT_DIR = APPROACH_ROOT / "inputs" / STAGE_NAME

OFFICIAL_EXTERNAL_METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / DATASET3_NAME / "external_classification"
OFFICIAL_EXTERNAL_FEATURES_DIR = PROJECT_ROOT / "data" / "features" / DATASET3_NAME / "external_classification"
SMOKE_EXTERNAL_METADATA_DIR = OUTPUT_DIR / "smoke_external_classification_metadata"
SMOKE_EXTERNAL_FEATURES_DIR = OUTPUT_DIR / "smoke_external_classification_features"
EXTERNAL_METADATA_DIR = SMOKE_EXTERNAL_METADATA_DIR if RUN_SMOKE_TEST else OFFICIAL_EXTERNAL_METADATA_DIR
EXTERNAL_FEATURES_DIR = SMOKE_EXTERNAL_FEATURES_DIR if RUN_SMOKE_TEST else OFFICIAL_EXTERNAL_FEATURES_DIR

for directory in [TABLES_DIR, FIGURES_DIR, PARTIAL_DIR, EXTERNAL_METADATA_DIR, EXTERNAL_FEATURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

EXPLORATION_TABLE_CANDIDATES = [
    APPROACH_ROOT / "outputs" / "01_exploration" / "03_dataset3_external_exploration" / "tables",
    APPROACH_ROOT / "outputs" / "01_exploration" / "03_dataset3_external_exploration" / "tables",
]
EXPLORATION_TABLES_DIR = next((candidate for candidate in EXPLORATION_TABLE_CANDIDATES if candidate.exists()), None)

SAVED_FILES = []


def timestamp_now():
    return datetime.now().isoformat(timespec="seconds")


def relative_path(path):
    path = Path(path)

    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def record_saved_file(path, file_type, action="created_or_overwritten", rows=None, columns=None, note=""):
    path = Path(path)
    size_mb = path.stat().st_size / (1024 * 1024) if path.exists() else np.nan
    SAVED_FILES.append({
        "file_path": relative_path(path),
        "file_type": file_type,
        "action": action,
        "rows": rows,
        "columns": columns,
        "file_size_mb": round(size_mb, 4) if not pd.isna(size_mb) else np.nan,
        "note": note,
        "timestamp": timestamp_now(),
    })


def save_csv(df, path, note="", overwrite=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite is None:
        overwrite = OVERWRITE_TABLES

    if path.exists() and not overwrite:
        print(f"[SKIP] Mevcut CSV korundu: {relative_path(path)}")
        try:
            existing = pd.read_csv(path)
            record_saved_file(path, "csv", "loaded_existing", len(existing), len(existing.columns), note)
            return existing
        except pd.errors.EmptyDataError:
            print(f"[INFO] Mevcut CSV boş olduğu için güncel tablo bellekte kullanılacak: {relative_path(path)}")
            return df

    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[SAVE] CSV kaydedildi: {relative_path(path)} | shape={df.shape}")
    record_saved_file(path, "csv", "created_or_overwritten", len(df), len(df.columns), note)
    return df


def save_parquet(df, path, note="", overwrite=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not overwrite:
        print(f"[SKIP] Mevcut parquet korundu: {relative_path(path)}")
        existing = pd.read_parquet(path)
        record_saved_file(path, "parquet", "loaded_existing", len(existing), len(existing.columns), note)
        return existing

    df.to_parquet(path, index=False)
    print(f"[SAVE] Parquet kaydedildi: {relative_path(path)} | shape={df.shape}")
    record_saved_file(path, "parquet", "created_or_overwritten", len(df), len(df.columns), note)
    return df


def log_dataframe(title, df, n=20):
    print()
    print(f"[{title}] shape={df.shape}")
    display(df.head(n))


print(f"[PATH] PROJECT_ROOT={PROJECT_ROOT}")
print(f"[PATH] DATASET3_TEST_IMAGES_DIR={DATASET3_TEST_IMAGES_DIR}")
print(f"[PATH] DATASET3_TEST_LABELS_DIR={DATASET3_TEST_LABELS_DIR}")
print(f"[PATH] EXPLORATION_TABLES_DIR={EXPLORATION_TABLES_DIR}")
print(f"[PATH] RUN_MODE={RUN_MODE}")
print(f"[PATH] OUTPUT_DIR={OUTPUT_DIR}")
print(f"[PATH] EXTERNAL_METADATA_DIR={EXTERNAL_METADATA_DIR}")
print(f"[PATH] EXTERNAL_FEATURES_DIR={EXTERNAL_FEATURES_DIR}")
print(f"[PATH] INPUT_DIR={INPUT_DIR}")


## 4. Finalist Modeller

Bu bölümde dış doğrulamada kullanılacak sabit finalist model listesi ve detection parametreleri tanımlanır.


In [ ]:
FINALIST_ROWS = [
    dict(source_dataset="dataset1_varroa", source_group="Dataset1-derived", algorithm_name="linear_svm",
         model_id="d1_linsvm_011", model_file_name="d1_linsvm_011__centered80_balanced__hog16_pca_hsv_lbp.joblib",
         patch_set="centered80_balanced", patch_size=80, feature_set="hog16_pca_hsv_lbp",
         threshold_column="decision_function_score_threshold", threshold_value=2.20, stride_divisor=4,
         cluster_iou_threshold=0.20, weighted_box_size_factor=1.00),
    dict(source_dataset="dataset1_varroa", source_group="Dataset1-derived", algorithm_name="rbf_svm",
         model_id="d1_rbfsvm_006", model_file_name="d1_rbfsvm_006__centered80_neg4x__hog16_hsv_lbp.joblib",
         patch_set="centered80_neg4x", patch_size=80, feature_set="hog16_hsv_lbp",
         threshold_column="decision_function_score_threshold", threshold_value=1.20, stride_divisor=4,
         cluster_iou_threshold=0.25, weighted_box_size_factor=1.00),
    dict(source_dataset="dataset2_honeybee", source_group="Dataset2-derived", algorithm_name="random_forest",
         model_id="d2_rf_007", model_file_name="d2_rf_007__centered48_neg4x__hog16_pca_hsv_lbp.joblib",
         patch_set="centered48_neg4x", patch_size=48, feature_set="hog16_pca_hsv_lbp",
         threshold_column="predict_proba_score_threshold", threshold_value=0.70, stride_divisor=4,
         cluster_iou_threshold=0.20, weighted_box_size_factor=1.00),
    dict(source_dataset="dataset2_honeybee", source_group="Dataset2-derived", algorithm_name="random_forest",
         model_id="d2_rf_006", model_file_name="d2_rf_006__centered48_neg4x__hog16_hsv_lbp.joblib",
         patch_set="centered48_neg4x", patch_size=48, feature_set="hog16_hsv_lbp",
         threshold_column="predict_proba_score_threshold", threshold_value=0.60, stride_divisor=4,
         cluster_iou_threshold=0.20, weighted_box_size_factor=1.00),
    dict(source_dataset="dataset2_honeybee", source_group="Dataset2-derived", algorithm_name="rbf_svm",
         model_id="d2_rbfsvm_007", model_file_name="d2_rbfsvm_007__centered48_neg4x__hog16_pca_hsv_lbp.joblib",
         patch_set="centered48_neg4x", patch_size=48, feature_set="hog16_pca_hsv_lbp",
         threshold_column="decision_function_score_threshold", threshold_value=0.90, stride_divisor=4,
         cluster_iou_threshold=0.20, weighted_box_size_factor=1.00),
]

finalists_df = pd.DataFrame(FINALIST_ROWS)

FINALIST_CSV_CANDIDATES = [
    INPUT_DIR / "stage11_external_finalist_models.csv",
]
FINALIST_CSV_PATH = next((path for path in FINALIST_CSV_CANDIDATES if path.exists()), None)


def audit_finalist_csv(finalists_df, csv_path):
    audit_rows = []

    if csv_path is None:
        audit_rows.append({
            "check_name": "stage11_external_finalist_models_csv_found",
            "passed": False,
            "observed": "not found",
            "expected": "approaches/traditional_ml/inputs/11_external_validation/stage11_external_finalist_models.csv",
            "severity": "WARNING",
        })
        return pd.DataFrame(audit_rows)

    csv_df = pd.read_csv(csv_path)

    audit_rows.append({
        "check_name": "stage11_external_finalist_models_csv_found",
        "passed": True,
        "observed": relative_path(csv_path),
        "expected": "csv available",
        "severity": "INFO",
    })
    audit_rows.append({
        "check_name": "csv_model_count",
        "passed": len(csv_df) == len(finalists_df),
        "observed": len(csv_df),
        "expected": len(finalists_df),
        "severity": "ERROR",
    })

    expected_by_id = finalists_df.set_index("model_id")
    actual_by_id = csv_df.set_index("model_id") if "model_id" in csv_df.columns else pd.DataFrame()

    audit_rows.append({
        "check_name": "csv_model_ids_match",
        "passed": set(expected_by_id.index) == set(actual_by_id.index),
        "observed": sorted(actual_by_id.index.tolist()) if not actual_by_id.empty else [],
        "expected": sorted(expected_by_id.index.tolist()),
        "severity": "ERROR",
    })

    column_map = {
        "source_dataset": "source_dataset",
        "model_file_name": "model_file_name",
        "algorithm_name": "algorithm_name",
        "patch_set": "patch_set",
        "patch_size": "patch_size",
        "feature_set": "feature_set",
        "threshold_value": "detection_threshold",
        "stride_divisor": "stride_divisor",
        "cluster_iou_threshold": "cluster_iou_threshold",
        "weighted_box_size_factor": "weighted_box_size_factor",
    }

    for model_id in sorted(set(expected_by_id.index) & set(actual_by_id.index)):
        for expected_col, actual_col in column_map.items():
            if actual_col not in actual_by_id.columns:
                audit_rows.append({
                    "check_name": f"csv_column_exists::{actual_col}",
                    "passed": False,
                    "observed": "missing",
                    "expected": "present",
                    "severity": "ERROR",
                })
                continue

            expected_value = expected_by_id.loc[model_id, expected_col]
            observed_value = actual_by_id.loc[model_id, actual_col]

            if expected_col in {
                "threshold_value",
                "stride_divisor",
                "cluster_iou_threshold",
                "weighted_box_size_factor",
                "patch_size",
            }:
                passed = math.isclose(float(expected_value), float(observed_value), rel_tol=1e-9, abs_tol=1e-9)
            else:
                passed = str(expected_value) == str(observed_value)

            audit_rows.append({
                "check_name": f"{model_id}::{expected_col}",
                "passed": passed,
                "observed": observed_value,
                "expected": expected_value,
                "severity": "ERROR",
            })

    return pd.DataFrame(audit_rows)


finalist_csv_audit_df = audit_finalist_csv(finalists_df, FINALIST_CSV_PATH)
save_csv(finalist_csv_audit_df, TABLES_DIR / "stage11_finalist_csv_audit.csv", "Stage 11 finalist CSV audit")
log_dataframe("Stage 11 finalist CSV audit", finalist_csv_audit_df)

if FINALIST_CSV_PATH is not None and not finalist_csv_audit_df[finalist_csv_audit_df["severity"] == "ERROR"]["passed"].all():
    raise RuntimeError("Stage 11 finalist CSV audit failed. Fix finalist list or fixed parameters before continuing.")


def resolve_model_path(row):
    candidates = [
        PROJECT_ROOT / "outputs" / "models" / "selected_classification_model_pool" / row["source_dataset"] / row["algorithm_name"] / row["model_file_name"],
        PROJECT_ROOT / "outputs" / "models" / row["source_dataset"] / "01_linear_svm" / "models" / row["model_file_name"],
        PROJECT_ROOT / "outputs" / "models" / row["source_dataset"] / "00_linear_svm" / "models" / row["model_file_name"],
        PROJECT_ROOT / "outputs" / "models" / row["source_dataset"] / "02_selected_rbf_svm" / "models" / row["model_file_name"],
        PROJECT_ROOT / "outputs" / "models" / row["source_dataset"] / "03_selected_random_forest" / "models" / row["model_file_name"],
    ]

    for path in candidates:
        if path.exists():
            return str(path)

    return str(candidates[0])


finalists_df["model_path_resolved"] = finalists_df.apply(resolve_model_path, axis=1)
finalists_df["model_path_exists"] = finalists_df["model_path_resolved"].apply(lambda path: Path(path).exists())

finalists_export_df = finalists_df.copy()
finalists_export_df["model_path_resolved"] = finalists_export_df["model_path_resolved"].map(relative_path)

save_csv(finalists_export_df, TABLES_DIR / "stage11_finalist_models.csv", "Fixed Stage 11 finalist models")
save_csv(
    finalists_df[[
        "model_id",
        "model_file_name",
        "threshold_value",
        "stride_divisor",
        "cluster_iou_threshold",
        "weighted_box_size_factor",
    ]],
    TABLES_DIR / "stage11_fixed_detection_parameters.csv",
    "Fixed Stage 11 detection parameters",
    OVERWRITE_TABLES,
)

log_dataframe("Fixed finalists", finalists_df)

if not finalists_df["model_path_exists"].all():
    warnings.warn("Some model artifacts were not found. Fix model paths/artifacts before full execution.")

## 5. Dataset3 Envanteri ve Exploration Kontrolü

Bu bölümde Dataset3 test görüntüleri, etiketleri ve varsa önceki exploration tabloları kontrol edilir.


In [ ]:
def read_exploration_table(file_name):
    if EXPLORATION_TABLES_DIR is None:
        return pd.DataFrame()
    p = EXPLORATION_TABLES_DIR / file_name
    return pd.read_csv(p) if p.exists() else pd.DataFrame()

def add_audit(rows, name, passed, observed, expected, severity="ERROR"):
    rows.append(dict(check_name=name, passed=bool(passed), observed=observed, expected=expected, severity=severity))

audit_rows = []
if RUN_EXPLORATION_AUDIT:
    if EXPLORATION_TABLES_DIR is None:
        add_audit(audit_rows, "exploration_tables_dir_found", False, "not found", "existing tables directory", "WARNING")
    else:
        file_counts = read_exploration_table("dataset_file_count_summary.csv")
        class_dist = read_exploration_table("class_distribution_by_split.csv")
        obj_summary = read_exploration_table("object_count_per_image_summary.csv")
        img_summary = read_exploration_table("image_size_summary.csv")
        final_status = read_exploration_table("final_status_summary.csv")

        if not file_counts.empty:
            test = file_counts[file_counts["split"].astype(str) == "test"]
            add_audit(audit_rows, "test_image_count", len(test)==1 and int(test.iloc[0]["image_count"])==596, int(test.iloc[0]["image_count"]) if len(test) else None, 596)
            add_audit(audit_rows, "test_label_count", len(test)==1 and int(test.iloc[0]["label_count"])==596, int(test.iloc[0]["label_count"]) if len(test) else None, 596)
            add_audit(audit_rows, "test_image_label_delta", len(test)==1 and int(test.iloc[0]["image_minus_label_count"])==0, int(test.iloc[0]["image_minus_label_count"]) if len(test) else None, 0)

        if not class_dist.empty:
            test = class_dist[class_dist["split"].astype(str) == "test"]
            add_audit(audit_rows, "test_class_ids", set(test["class_id"].astype(int)) == {0}, sorted(test["class_id"].astype(int).unique().tolist()), "{0}")
            add_audit(audit_rows, "test_gt_objects", int(test["object_count"].sum()) == 296, int(test["object_count"].sum()), 296)

        if not obj_summary.empty:
            test = obj_summary[obj_summary["split"].astype(str) == "test"]
            if len(test)==1:
                r = test.iloc[0]
                add_audit(audit_rows, "test_zero_images", int(r["images_with_zero_varroa"])==358, int(r["images_with_zero_varroa"]), 358)
                add_audit(audit_rows, "test_positive_images", int(r["images_with_varroa"])==238, int(r["images_with_varroa"]), 238)
                add_audit(audit_rows, "test_total_varroa_objects", int(r["total_varroa_objects"])==296, int(r["total_varroa_objects"]), 296)

        if not img_summary.empty:
            test = img_summary[img_summary["split"].astype(str) == "test"]
            if len(test)==1:
                r = test.iloc[0]
                ok = int(r["min_width"])==640 and int(r["max_width"])==640 and int(r["min_height"])==640 and int(r["max_height"])==640
                add_audit(audit_rows, "test_image_size_640", ok, f"{r['min_width']}..{r['max_width']} x {r['min_height']}..{r['max_height']}", "640x640")

        if not final_status.empty:
            r = final_status.iloc[0]
            for col in ["missing_label_count", "missing_image_count", "unreadable_image_count", "invalid_yolo_row_count", "varroa_bbox_exclusion_candidate_count", "image_exclusion_candidate_count"]:
                add_audit(audit_rows, col, int(r[col])==0, int(r[col]), 0)

audit_df = pd.DataFrame(audit_rows)
save_csv(audit_df, TABLES_DIR / "dataset3_exploration_audit.csv", "Dataset3 exploration audit")
log_dataframe("Exploration audit", audit_df, 50)

if not audit_df.empty:
    hard_fail = audit_df[(audit_df["severity"]=="ERROR") & (~audit_df["passed"])]
    if not hard_fail.empty:
        raise RuntimeError("Exploration audit failed. Review dataset3_exploration_audit.csv before continuing.")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_image_files(images_dir):
    return sorted([p for p in Path(images_dir).rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS])

def parse_yolo_label_file(label_path, image_width, image_height):
    rows = []
    label_path = Path(label_path)
    if not label_path.exists():
        return rows
    for row_index, line in enumerate(label_path.read_text(encoding="utf-8").splitlines()):
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"Invalid YOLO row in {label_path}: {line}")
        class_id = int(float(parts[0]))
        if class_id not in DATASET3_POSITIVE_CLASS_IDS:
            continue
        xcn, ycn, wn, hn = map(float, parts[1:])
        cx, cy = xcn * image_width, ycn * image_height
        bw, bh = wn * image_width, hn * image_height
        rows.append(dict(row_index=row_index, class_id=class_id, class_name="varroa-mite",
                         x_center_norm=xcn, y_center_norm=ycn, bbox_width_norm=wn, bbox_height_norm=hn,
                         bbox_x_center_px=cx, bbox_y_center_px=cy, bbox_width_px=bw, bbox_height_px=bh,
                         bbox_area_px=bw*bh, x1=cx-bw/2, y1=cy-bh/2, x2=cx+bw/2, y2=cy+bh/2))
    return rows

def read_image_shape(image_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    h, w = img.shape[:2]
    return w, h

def build_dataset3_test_records():
    if not DATASET3_TEST_IMAGES_DIR.exists():
        raise FileNotFoundError(DATASET3_TEST_IMAGES_DIR)
    if not DATASET3_TEST_LABELS_DIR.exists():
        raise FileNotFoundError(DATASET3_TEST_LABELS_DIR)
    image_paths = find_image_files(DATASET3_TEST_IMAGES_DIR)
    records, label_rows = [], []
    for image_path in image_paths:
        image_id = image_path.stem
        label_path = DATASET3_TEST_LABELS_DIR / f"{image_id}.txt"
        w, h = read_image_shape(image_path)
        gt_rows = parse_yolo_label_file(label_path, w, h)
        gt_boxes = []
        for gt in gt_rows:
            gt_boxes.append({k: gt[k] for k in ["x1", "y1", "x2", "y2"]})
        label_rows.append(dict(dataset_name=DATASET3_NAME, split="test", image_id=image_id,
                            source_image_path=relative_path(image_path), source_label_path=relative_path(label_path),
                            image_width=w, image_height=h, **gt))
        records.append(dict(dataset_name=DATASET3_NAME, split="test", image_id=image_id,
                            image_path=relative_path(image_path), label_path=relative_path(label_path),
                            image_width=w, image_height=h, gt_boxes=gt_boxes, gt_count=len(gt_boxes)))
    return pd.DataFrame(records), pd.DataFrame(label_rows)

dataset3_image_records_df, dataset3_gt_labels_df = build_dataset3_test_records()

if RUN_SMOKE_TEST:
    pos = dataset3_image_records_df[dataset3_image_records_df["gt_count"] > 0].head(max(1, SMOKE_TEST_IMAGE_LIMIT // 2))
    neg = dataset3_image_records_df[dataset3_image_records_df["gt_count"] == 0].head(max(1, SMOKE_TEST_IMAGE_LIMIT - len(pos)))
    dataset3_eval_records_df = pd.concat([pos, neg], ignore_index=True).head(SMOKE_TEST_IMAGE_LIMIT)
else:
    dataset3_eval_records_df = dataset3_image_records_df.copy()

inventory_df = pd.DataFrame([dict(dataset_name=DATASET3_NAME, split="test",
                                  raw_test_image_count=len(dataset3_image_records_df),
                                  evaluation_image_count=len(dataset3_eval_records_df),
                                  zero_image_count=int((dataset3_image_records_df["gt_count"]==0).sum()),
                                  positive_image_count=int((dataset3_image_records_df["gt_count"]>0).sum()),
                                  gt_object_count=int(dataset3_image_records_df["gt_count"].sum()),
                                  run_smoke_test=RUN_SMOKE_TEST)])
dist_df = dataset3_image_records_df["gt_count"].value_counts().sort_index().reset_index()
dist_df.columns = ["gt_count", "image_count"]
dist_df["count_group"] = dist_df["gt_count"].apply(lambda x: "zero" if x==0 else ("one" if x==1 else ("two" if x==2 else "three_plus")))

save_csv(inventory_df, TABLES_DIR / "dataset3_external_test_inventory.csv", "Dataset3 test inventory")
save_csv(dist_df, TABLES_DIR / "dataset3_external_distribution_summary.csv", "Dataset3 test distribution")
save_csv(dataset3_gt_labels_df, TABLES_DIR / "dataset3_external_gt_labels.csv", "Parsed Dataset3 GT labels")
log_dataframe("Dataset3 inventory", inventory_df)
log_dataframe("Dataset3 GT-count distribution", dist_df)


## 6. Patch, Feature ve Detection Yardımcıları

Bu bölümde external classification metadata üretimi, feature çıkarma, model skorlaması ve detection yardımcı fonksiyonları tanımlanır.


In [ ]:
PATCH_METADATA_COLUMNS = [
    "patch_id","dataset_name","split","source_image_path","source_label_path","patch_set","patch_size","ratio_variant",
    "patch_label","patch_type","x1","y1","x2","y2","crop_width","crop_height","image_width","image_height",
    "patch_center_x_px","patch_center_y_px","requested_center_x_px","requested_center_y_px","was_shifted_to_fit_image",
    "shift_x_px","shift_y_px","source_bbox_id","source_bbox_row_index","source_bbox_class_id","source_bbox_class_name",
    "source_bbox_x_center_norm","source_bbox_y_center_norm","source_bbox_width_norm","source_bbox_height_norm",
    "source_bbox_x_center_px","source_bbox_y_center_px","source_bbox_width_px","source_bbox_height_px","source_bbox_area_px",
    "negative_type","requested_negative_zone","actual_negative_zone","actual_normalized_center_distance",
    "negative_source_mode","max_iou_with_any_gt","overlaps_varroa_bbox","sampling_seed","sampling_attempt_index","notes","flags"
]

def safe_divide(a, b):
    return float(a / b) if b else 0.0

def compute_iou(a, b):
    ix1, iy1 = max(float(a["x1"]), float(b["x1"])), max(float(a["y1"]), float(b["y1"]))
    ix2, iy2 = min(float(a["x2"]), float(b["x2"])), min(float(a["y2"]), float(b["y2"]))
    iw, ih = max(0.0, ix2-ix1), max(0.0, iy2-iy1)
    inter = iw * ih
    aa = max(0.0, float(a["x2"])-float(a["x1"])) * max(0.0, float(a["y2"])-float(a["y1"]))
    ab = max(0.0, float(b["x2"])-float(b["x1"])) * max(0.0, float(b["y2"])-float(b["y1"]))
    union = aa + ab - inter
    return inter / union if union > 0 else 0.0

def compute_crop_from_center(cx, cy, patch_size, image_width, image_height):
    if patch_size > image_width or patch_size > image_height:
        return None
    rx1, ry1 = int(round(cx - patch_size/2)), int(round(cy - patch_size/2))
    x1 = max(0, min(rx1, int(image_width - patch_size)))
    y1 = max(0, min(ry1, int(image_height - patch_size)))
    x2, y2 = x1 + patch_size, y1 + patch_size
    pcx, pcy = x1 + patch_size/2, y1 + patch_size/2
    return dict(x1=int(x1), y1=int(y1), x2=int(x2), y2=int(y2), crop_width=int(patch_size), crop_height=int(patch_size),
                patch_center_x_px=float(pcx), patch_center_y_px=float(pcy), requested_center_x_px=float(cx), requested_center_y_px=float(cy),
                was_shifted_to_fit_image=bool(x1 != rx1 or y1 != ry1), shift_x_px=int(x1-rx1), shift_y_px=int(y1-ry1))

def normalized_center_distance(cx, cy, w, h):
    denom = math.sqrt((w/2)**2 + (h/2)**2)
    return float(math.sqrt((cx-w/2)**2 + (cy-h/2)**2) / denom) if denom else 0.0

def zone_from_distance(d):
    return "center" if d < 0.25 else ("mixed" if d < 0.50 else "outer")

def negative_type_to_zone(nt):
    return nt.replace("_negative", "")

def sample_center_for_zone(rng, w, h, zone):
    for _ in range(MAX_NEGATIVE_SAMPLING_ATTEMPTS):
        cx, cy = float(rng.uniform(0, w)), float(rng.uniform(0, h))
        if zone_from_distance(normalized_center_distance(cx, cy, w, h)) == zone:
            return cx, cy
    if zone == "center":
        return w/2, h/2
    if zone == "mixed":
        return w*0.70, h/2
    return w*0.08, h*0.08

def max_iou_with_gts(crop, gt_boxes):
    return float(max([compute_iou(crop, gt) for gt in gt_boxes], default=0.0))

gt_by_image_path = {str(r["image_path"]): r["gt_boxes"] for _, r in dataset3_image_records_df.iterrows()}

def make_patch_id(patch_set, patch_type, idx):
    return f"{DATASET3_NAME}__{patch_set}__test__{patch_type}__{idx:08d}"

def build_positive_patch_rows(patch_set, patch_size, ratio_variant, max_positive_patches=None):
    rows = []
    labels = dataset3_gt_labels_df.copy()
    if max_positive_patches is not None:
        labels = labels.head(max_positive_patches)
    for idx, r in labels.reset_index(drop=True).iterrows():
        crop = compute_crop_from_center(r["bbox_x_center_px"], r["bbox_y_center_px"], patch_size, r["image_width"], r["image_height"])
        if crop is None:
            continue
        row = dict(patch_id=make_patch_id(patch_set, "positive", idx+1), dataset_name=DATASET3_NAME, split="test",
                   source_image_path=r["source_image_path"], source_label_path=r["source_label_path"], patch_set=patch_set,
                   patch_size=patch_size, ratio_variant=ratio_variant, patch_label=1, patch_type="positive",
                   image_width=int(r["image_width"]), image_height=int(r["image_height"]),
                   source_bbox_id=f"{DATASET3_NAME}__test__{Path(r['source_image_path']).stem}__row{int(r['row_index'])}",
                   source_bbox_row_index=int(r["row_index"]), source_bbox_class_id=int(r["class_id"]), source_bbox_class_name=r["class_name"],
                   source_bbox_x_center_norm=r["x_center_norm"], source_bbox_y_center_norm=r["y_center_norm"],
                   source_bbox_width_norm=r["bbox_width_norm"], source_bbox_height_norm=r["bbox_height_norm"],
                   source_bbox_x_center_px=r["bbox_x_center_px"], source_bbox_y_center_px=r["bbox_y_center_px"],
                   source_bbox_width_px=r["bbox_width_px"], source_bbox_height_px=r["bbox_height_px"], source_bbox_area_px=r["bbox_area_px"],
                   negative_type="", requested_negative_zone="", actual_negative_zone="", actual_normalized_center_distance=np.nan,
                   negative_source_mode="", max_iou_with_any_gt=np.nan, overlaps_varroa_bbox=np.nan, sampling_seed=RANDOM_SEED,
                   sampling_attempt_index=0, notes="", flags="")
        row.update(crop)
        rows.append(row)
    return rows

def build_negative_patch_rows(patch_set, patch_size, ratio_variant, negative_total):
    sampling_seed = RANDOM_SEED + PATCH_SET_SEED_OFFSETS.get(patch_set, 999)
    rng = np.random.default_rng(sampling_seed)
    image_pool = dataset3_image_records_df.reset_index(drop=True)
    rows, patch_index = [], 1
    counts = {nt: negative_total // len(NEGATIVE_TYPES) for nt in NEGATIVE_TYPES}
    for nt in NEGATIVE_TYPES[:negative_total % len(NEGATIVE_TYPES)]:
        counts[nt] += 1
    for nt, target in counts.items():
        zone = negative_type_to_zone(nt)
        attempts = 0
        while sum(x["negative_type"] == nt for x in rows) < target:
            attempts += 1
            if attempts > max(1, target) * MAX_NEGATIVE_SAMPLING_ATTEMPTS:
                raise RuntimeError(f"Could not sample enough negatives for {patch_set}/{nt}")
            ir = image_pool.iloc[int(rng.integers(0, len(image_pool)))]
            w, h = int(ir["image_width"]), int(ir["image_height"])
            cx, cy = sample_center_for_zone(rng, w, h, zone)
            crop = compute_crop_from_center(cx, cy, patch_size, w, h)
            if crop is None:
                continue
            max_iou = max_iou_with_gts(crop, gt_by_image_path.get(str(ir["image_path"]), []))
            if max_iou > NEGATIVE_MAX_IOU_WITH_ANY_GT:
                continue
            dist = normalized_center_distance(crop["patch_center_x_px"], crop["patch_center_y_px"], w, h)
            row = dict(patch_id=make_patch_id(patch_set, "negative", patch_index), dataset_name=DATASET3_NAME, split="test",
                       source_image_path=ir["image_path"], source_label_path=ir["label_path"], patch_set=patch_set,
                       patch_size=patch_size, ratio_variant=ratio_variant, patch_label=0, patch_type="negative",
                       image_width=w, image_height=h, source_bbox_id="", source_bbox_row_index=np.nan, source_bbox_class_id=np.nan,
                       source_bbox_class_name="", source_bbox_x_center_norm=np.nan, source_bbox_y_center_norm=np.nan,
                       source_bbox_width_norm=np.nan, source_bbox_height_norm=np.nan, source_bbox_x_center_px=np.nan,
                       source_bbox_y_center_px=np.nan, source_bbox_width_px=np.nan, source_bbox_height_px=np.nan,
                       source_bbox_area_px=np.nan, negative_type=nt, requested_negative_zone=zone, actual_negative_zone=zone_from_distance(dist),
                       actual_normalized_center_distance=dist, negative_source_mode="dataset3_test_non_overlap",
                       max_iou_with_any_gt=max_iou, overlaps_varroa_bbox=False, sampling_seed=sampling_seed,
                       sampling_attempt_index=attempts, notes="", flags="shifted_to_fit_image" if crop["was_shifted_to_fit_image"] else "")
            row.update(crop)
            rows.append(row)
            patch_index += 1
    return rows

image_cache = OrderedDict()
PCA_CACHE, MODEL_CACHE = {}, {}

def read_image_cached(path):
    path = Path(path)
    if path in image_cache:
        image_cache.move_to_end(path)
        return image_cache[path]
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    image_cache[path] = img
    if len(image_cache) > IMAGE_CACHE_MAX_ITEMS:
        image_cache.popitem(last=False)
    return img

def crop_patch_from_row(r):
    img = read_image_cached(r["source_image_path"])
    x1, y1, x2, y2 = int(r["x1"]), int(r["y1"]), int(r["x2"]), int(r["y2"])
    patch = img[y1:y2, x1:x2]
    expected = int(r["patch_size"])
    if patch.shape[:2] != (expected, expected):
        raise ValueError(f"Unexpected patch shape: {patch.shape[:2]} expected {expected}")
    return patch

def extract_hog_features(patch, hog_variant):
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    return hog(gray, orientations=HOG_ORIENTATIONS, pixels_per_cell=HOG_CONFIGS[hog_variant]["pixels_per_cell"],
               cells_per_block=HOG_CELLS_PER_BLOCK, block_norm=HOG_BLOCK_NORM,
               transform_sqrt=HOG_TRANSFORM_SQRT, feature_vector=True).astype(np.float32)

def extract_hsv_histogram(patch):
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    out = []
    for channel, bins, rng in [(0, HSV_H_BINS, [0,180]), (1, HSV_S_BINS, [0,256]), (2, HSV_V_BINS, [0,256])]:
        hist = cv2.calcHist([hsv], [channel], None, [bins], rng).flatten()
        out.append((hist / hist.sum()).astype(np.float32) if hist.sum() > 0 else np.zeros_like(hist, dtype=np.float32))
    return np.concatenate(out).astype(np.float32)

def extract_lbp_histogram(patch):
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, P=LBP_POINTS, R=LBP_RADIUS, method=LBP_METHOD)
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, LBP_POINTS + 3), range=(0, LBP_POINTS + 2))
    return (hist / hist.sum()).astype(np.float32) if hist.sum() > 0 else np.zeros(LBP_POINTS + 2, dtype=np.float32)

def make_hsv_columns():
    return [f"hsv_h_{i:02d}" for i in range(HSV_H_BINS)] + [f"hsv_s_{i:02d}" for i in range(HSV_S_BINS)] + [f"hsv_v_{i:02d}" for i in range(HSV_V_BINS)]

def make_lbp_columns():
    return [f"lbp_{i:02d}" for i in range(LBP_POINTS + 2)]

def load_source_pca(source_dataset, patch_set, hog_variant):
    key = (source_dataset, patch_set, hog_variant)
    if key in PCA_CACHE:
        return PCA_CACHE[key]
    p = PROJECT_ROOT / "data" / "features" / source_dataset / "pca_artifacts" / f"{patch_set}__{hog_variant}_pca.joblib"
    if not p.exists():
        raise FileNotFoundError(f"Missing PCA artifact: {p}")
    payload = joblib.load(p)
    pca = payload["pca"] if isinstance(payload, dict) and "pca" in payload else payload
    PCA_CACHE[key] = pca
    return pca

def extract_feature_matrix_from_patches(patches, source_dataset, patch_set, feature_set):
    cfg = FEATURE_SET_CONFIG[feature_set]
    comps = []
    hv = cfg["hog_variant"]
    if hv:
        hm = np.vstack([extract_hog_features(p, hv) for p in patches]).astype(np.float32)
        comps.append(load_source_pca(source_dataset, patch_set, hv).transform(hm).astype(np.float32) if cfg["use_pca"] else hm)
    if cfg["include_hsv"]:
        comps.append(np.vstack([extract_hsv_histogram(p) for p in patches]).astype(np.float32))
    if cfg["include_lbp"]:
        comps.append(np.vstack([extract_lbp_histogram(p) for p in patches]).astype(np.float32))
    return np.concatenate(comps, axis=1).astype(np.float32)

def build_feature_dataframe(metadata_df, feature_set, source_dataset):
    patches = [crop_patch_from_row(r) for _, r in metadata_df.iterrows()]
    X = extract_feature_matrix_from_patches(patches, source_dataset, str(metadata_df["patch_set"].iloc[0]), feature_set)
    cfg = FEATURE_SET_CONFIG[feature_set]
    cols = []
    if cfg["hog_variant"]:
        if cfg["use_pca"]:
            tail_dim = 0
            if cfg["include_hsv"]:
                tail_dim += HSV_H_BINS + HSV_S_BINS + HSV_V_BINS
            if cfg["include_lbp"]:
                tail_dim += LBP_POINTS + 2
            pca_dim = int(X.shape[1] - tail_dim)
            cols += [f"{cfg['hog_variant']}_pca_{i:04d}" for i in range(pca_dim)]
        else:
            hog_len = extract_hog_features(patches[0], cfg["hog_variant"]).shape[0]
            cols += [f"{cfg['hog_variant']}_{i:04d}" for i in range(hog_len)]
    if cfg["include_hsv"]:
        cols += make_hsv_columns()
    if cfg["include_lbp"]:
        cols += make_lbp_columns()
    # Fallback if PCA dimension calculation differs.
    if len(cols) != X.shape[1]:
        cols = [f"feature_{i:05d}" for i in range(X.shape[1])]
    meta = metadata_df.copy().reset_index(drop=True)
    for col in PATCH_METADATA_COLUMNS:
        if col not in meta.columns:
            meta[col] = pd.NA
    return pd.concat([meta[PATCH_METADATA_COLUMNS], pd.DataFrame(X, columns=cols)], axis=1)

def unwrap_model_artifact(payload):
    if isinstance(payload, dict):
        for k in ["model", "pipeline", "estimator", "best_estimator"]:
            if k in payload:
                return payload[k]
    return payload

def load_model(path):
    path = Path(path)
    if path in MODEL_CACHE:
        return MODEL_CACHE[path]
    obj = unwrap_model_artifact(joblib.load(path))
    MODEL_CACHE[path] = obj
    return obj

def score_model(model, X, threshold_column=None):
    if threshold_column == "predict_proba_score_threshold" and hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X)[:, 1]).ravel()
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(X)).ravel()
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X)[:, 1]).ravel()
    raise AttributeError("Model exposes neither decision_function nor predict_proba.")

def predict_model(model, X):
    return np.asarray(model.predict(X)).astype(int)

def get_feature_columns(df):
    return [c for c in df.columns if c not in PATCH_METADATA_COLUMNS]


## 7. External Classification

Bu bölümde Dataset3 external classification patch metadata, feature tabloları ve finalist model classification sonuçları üretilir.


In [ ]:
patch_metadata_manifest_rows, patch_metadata_summary_rows = [], []
if RUN_EXTERNAL_CLASSIFICATION_METADATA:
    for patch_set, cfg in EXTERNAL_PATCH_SET_CONFIGS.items():
        out = EXTERNAL_METADATA_DIR / f"patch_metadata_{patch_set}.csv"
        if out.exists() and not OVERWRITE_EXTERNAL_PATCH_METADATA:
            df = pd.read_csv(out)
            action = "loaded_existing"
            record_saved_file(out, "csv", action, len(df), len(df.columns), f"External patch metadata {patch_set}")
        else:
            max_pos = SMOKE_TEST_MAX_POSITIVE_PATCHES if RUN_SMOKE_TEST else None
            pos_rows = build_positive_patch_rows(patch_set, cfg["patch_size"], cfg["ratio_variant"], max_pos)
            neg_rows = build_negative_patch_rows(patch_set, cfg["patch_size"], cfg["ratio_variant"], len(pos_rows) * cfg["negative_multiplier"])
            df = pd.DataFrame(pos_rows + neg_rows)
            for col in PATCH_METADATA_COLUMNS:
                if col not in df.columns:
                    df[col] = pd.NA
            df = df[PATCH_METADATA_COLUMNS]
            save_csv(df, out, f"External patch metadata {patch_set}", overwrite=True)
            action = "created_or_overwritten"
        patch_metadata_manifest_rows.append(dict(patch_set=patch_set, patch_metadata_path=relative_path(out), row_count=len(df),
                                                 positive_count=int((df["patch_label"]==1).sum()), negative_count=int((df["patch_label"]==0).sum()),
                                                 action=action, run_smoke_test=RUN_SMOKE_TEST))
        patch_metadata_summary_rows.append(dict(patch_set=patch_set, patch_size=cfg["patch_size"], ratio_variant=cfg["ratio_variant"],
                                                negative_multiplier=cfg["negative_multiplier"], row_count=len(df),
                                                positive_count=int((df["patch_label"]==1).sum()), negative_count=int((df["patch_label"]==0).sum())))

patch_metadata_manifest_df = pd.DataFrame(patch_metadata_manifest_rows)
patch_metadata_summary_df = pd.DataFrame(patch_metadata_summary_rows)
save_csv(patch_metadata_manifest_df, EXTERNAL_METADATA_DIR / "patch_metadata_manifest.csv", "External patch metadata manifest", overwrite=True)
save_csv(patch_metadata_summary_df, EXTERNAL_METADATA_DIR / "patch_metadata_summary.csv", "External patch metadata summary", overwrite=True)
save_csv(patch_metadata_manifest_df, TABLES_DIR / "external_classification_patch_metadata_manifest.csv", "External patch metadata manifest copy")
save_csv(patch_metadata_summary_df, TABLES_DIR / "patch_metadata_summary.csv", "External patch metadata summary copy")
log_dataframe("Patch metadata summary", patch_metadata_summary_df)

feature_plan_df = finalists_df[["source_dataset", "patch_set", "feature_set"]].drop_duplicates().reset_index(drop=True)
feature_manifest_rows = []
if RUN_EXTERNAL_CLASSIFICATION_FEATURES:
    for _, pr in feature_plan_df.iterrows():
        source_dataset, patch_set, feature_set = pr["source_dataset"], pr["patch_set"], pr["feature_set"]
        meta_path = EXTERNAL_METADATA_DIR / f"patch_metadata_{patch_set}.csv"
        out = EXTERNAL_FEATURES_DIR / f"{patch_set}__{feature_set}.parquet"
        if out.exists() and not OVERWRITE_EXTERNAL_FEATURES:
            fdf = pd.read_parquet(out)
            action = "loaded_existing"
            record_saved_file(out, "parquet", action, len(fdf), len(fdf.columns), f"External features {patch_set} {feature_set}")
        else:
            mdf = pd.read_csv(meta_path)
            fdf = build_feature_dataframe(mdf, feature_set, source_dataset)
            save_parquet(fdf, out, f"External features {patch_set} {feature_set}", True)
            action = "created_or_overwritten"
        fcols = get_feature_columns(fdf)
        feature_manifest_rows.append(dict(source_dataset_for_pca=source_dataset, patch_set=patch_set, feature_set=feature_set,
                                          feature_file_path=relative_path(out), row_count=len(fdf), feature_column_count=len(fcols),
                                          positive_count=int((fdf["patch_label"]==1).sum()), negative_count=int((fdf["patch_label"]==0).sum()),
                                          action=action, run_smoke_test=RUN_SMOKE_TEST))
        image_cache.clear()
        gc.collect()

external_feature_manifest_df = pd.DataFrame(feature_manifest_rows)
save_csv(external_feature_manifest_df, EXTERNAL_FEATURES_DIR / "external_feature_manifest.csv", "External feature manifest", overwrite=True)
save_csv(external_feature_manifest_df, EXTERNAL_FEATURES_DIR / "external_feature_summary.csv", "External feature summary", overwrite=True)
save_csv(external_feature_manifest_df, TABLES_DIR / "external_classification_feature_manifest.csv", "External feature manifest copy")
save_csv(external_feature_manifest_df, TABLES_DIR / "external_feature_summary.csv", "External feature summary copy")
log_dataframe("External feature manifest", external_feature_manifest_df)

def binary_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    tp = int(((y_true==1)&(y_pred==1)).sum()); fp = int(((y_true==0)&(y_pred==1)).sum())
    tn = int(((y_true==0)&(y_pred==0)).sum()); fn = int(((y_true==1)&(y_pred==0)).sum())
    precision = safe_divide(tp, tp+fp); recall = safe_divide(tp, tp+fn); specificity = safe_divide(tn, tn+fp)
    f1 = safe_divide(2*precision*recall, precision+recall)
    return dict(accuracy=safe_divide(tp+tn, tp+fp+tn+fn), precision=precision, recall=recall, specificity=specificity, f1=f1, tp=tp, fp=fp, tn=tn, fn=fn)

classification_rows, confusion_rows, score_rows = [], [], []
if RUN_EXTERNAL_CLASSIFICATION_EVALUATION:
    for _, mr in finalists_df.iterrows():
        fp = EXTERNAL_FEATURES_DIR / f"{mr['patch_set']}__{mr['feature_set']}.parquet"
        fdf = pd.read_parquet(fp)
        X = fdf[get_feature_columns(fdf)].to_numpy(dtype=np.float32, copy=False)
        y = fdf["patch_label"].astype(int).to_numpy()
        model = load_model(mr["model_path_resolved"])
        pred = predict_model(model, X)
        scores = score_model(model, X, mr["threshold_column"])
        metrics = binary_metrics(y, pred)
        pos_scores, neg_scores = scores[y==1], scores[y==0]
        row = dict(dataset_name=DATASET3_NAME, eval_component="external_classification_diagnostic",
                   source_dataset=mr["source_dataset"], source_group=mr["source_group"], algorithm_name=mr["algorithm_name"],
                   model_id=mr["model_id"], model_file_name=mr["model_file_name"], patch_set=mr["patch_set"], feature_set=mr["feature_set"],
                   row_count=len(fdf), positive_count=int((y==1).sum()), negative_count=int((y==0).sum()), **metrics,
                   positive_score_mean=float(np.mean(pos_scores)) if len(pos_scores) else np.nan,
                   negative_score_mean=float(np.mean(neg_scores)) if len(neg_scores) else np.nan,
                   score_separation_mean=float(np.mean(pos_scores)-np.mean(neg_scores)) if len(pos_scores) and len(neg_scores) else np.nan,
                   run_smoke_test=RUN_SMOKE_TEST)
        classification_rows.append(row)
        confusion_rows.append(dict(model_id=mr["model_id"], tp=metrics["tp"], fp=metrics["fp"], tn=metrics["tn"], fn=metrics["fn"]))
        score_rows.append({k: row[k] for k in ["model_id", "positive_score_mean", "negative_score_mean", "score_separation_mean"]})

external_classification_results_df = pd.DataFrame(classification_rows)
ranked_external_classification_results_df = external_classification_results_df.sort_values(["f1", "recall", "precision"], ascending=[False, False, False]) if not external_classification_results_df.empty else external_classification_results_df
save_csv(external_classification_results_df, TABLES_DIR / "external_classification_results.csv", "External classification results")
save_csv(ranked_external_classification_results_df, TABLES_DIR / "ranked_external_classification_results.csv", "Ranked external classification results")
save_csv(pd.DataFrame(confusion_rows), TABLES_DIR / "external_classification_confusion_matrices.csv", "External classification confusion matrices")
save_csv(pd.DataFrame(score_rows), TABLES_DIR / "external_classification_score_summary.csv", "External classification score summary")
log_dataframe("Ranked external classification results", ranked_external_classification_results_df)


## 8. External Detection

Bu bölümde finalist modeller Dataset3 test görüntüleri üzerinde sliding-window detector olarak değerlendirilir.


In [ ]:
def generate_positions(length, patch_size, stride):
    if length <= patch_size:
        return [0]
    pos = list(range(0, length - patch_size + 1, stride))
    last = length - patch_size
    if pos[-1] != last:
        pos.append(last)
    return sorted(set(int(x) for x in pos))

def generate_sliding_windows(w, h, patch_size, stride_divisor):
    stride = max(1, int(round(patch_size / stride_divisor)))
    windows = []
    for y in generate_positions(h, patch_size, stride):
        for x in generate_positions(w, patch_size, stride):
            windows.append(dict(x1=float(x), y1=float(y), x2=float(x+patch_size), y2=float(y+patch_size),
                                cx=float(x+patch_size/2), cy=float(y+patch_size/2)))
    return windows, stride

def score_image_windows(model, image, mr):
    h, w = image.shape[:2]
    windows, stride = generate_sliding_windows(w, h, int(mr["patch_size"]), int(mr["stride_divisor"]))
    scored, batch_patches, batch_windows = [], [], []
    def flush():
        nonlocal batch_patches, batch_windows, scored
        if not batch_patches:
            return
        X = extract_feature_matrix_from_patches(batch_patches, mr["source_dataset"], mr["patch_set"], mr["feature_set"])
        scores = score_model(model, X, mr["threshold_column"])
        scored.extend([{**win, "score": float(sc)} for win, sc in zip(batch_windows, scores)])
        batch_patches, batch_windows = [], []
    for win in windows:
        x1, y1, x2, y2 = map(int, [win["x1"], win["y1"], win["x2"], win["y2"]])
        patch = image[y1:y2, x1:x2]
        if patch.shape[:2] != (int(mr["patch_size"]), int(mr["patch_size"])):
            continue
        batch_patches.append(patch); batch_windows.append(win)
        if len(batch_patches) >= FEATURE_EXTRACTION_BATCH_SIZE:
            flush()
    flush()
    return scored, stride

def clip_box(box, w, h):
    box = dict(box)
    box["x1"] = max(0.0, min(float(w)-1.0, float(box["x1"])))
    box["y1"] = max(0.0, min(float(h)-1.0, float(box["y1"])))
    box["x2"] = max(0.0, min(float(w), float(box["x2"])))
    box["y2"] = max(0.0, min(float(h), float(box["y2"])))
    return box

def weighted_center_cluster(candidates, mr, image_width, image_height):
    remaining = sorted(candidates, key=lambda d: d["score"], reverse=True)
    final, eps = [], 1e-6
    while remaining:
        seed = remaining[0]
        cluster, keep = [seed], []
        for det in remaining[1:]:
            if compute_iou(seed, det) >= float(mr["cluster_iou_threshold"]):
                cluster.append(det)
            else:
                keep.append(det)
        scores = np.array([d["score"] for d in cluster], dtype=float)
        weights = np.maximum(scores - float(mr["threshold_value"]) + eps, eps)
        cx = float(np.sum(np.array([d["cx"] for d in cluster]) * weights) / np.sum(weights))
        cy = float(np.sum(np.array([d["cy"] for d in cluster]) * weights) / np.sum(weights))
        box_size = float(mr["patch_size"]) * float(mr["weighted_box_size_factor"])
        box = dict(x1=cx-box_size/2, y1=cy-box_size/2, x2=cx+box_size/2, y2=cy+box_size/2,
                   score=float(np.max(scores)), mean_score=float(np.mean(scores)), cluster_size=int(len(cluster)))
        final.append(clip_box(box, image_width, image_height))
        remaining = keep
    return sorted(final, key=lambda d: d["score"], reverse=True)

def run_detection_for_image(model, image_record, mr):
    image = cv2.imread(str(image_record["image_path"]), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(image_record["image_path"])
    start = time.time()
    scored, stride = score_image_windows(model, image, mr)
    candidates = sorted([w for w in scored if w["score"] >= float(mr["threshold_value"])], key=lambda d: d["score"], reverse=True)[:MAX_CANDIDATES_PER_IMAGE]
    detections = weighted_center_cluster(candidates, mr, int(image_record["image_width"]), int(image_record["image_height"]))
    return dict(image_id=image_record["image_id"], gt_boxes=image_record["gt_boxes"], detections=detections,
                scored_window_count=len(scored), candidate_count=len(candidates), final_detection_count=len(detections),
                stride_px=stride, runtime_seconds=time.time()-start)

def metric_key(x):
    return f"{x:.2f}".replace(".", "_")

def match_detections_to_gt(detections, gt_boxes, iou_threshold):
    detections = sorted(detections, key=lambda d: d.get("score", 0.0), reverse=True)
    matched, tp, fp, matched_ious = set(), 0, 0, []
    for det in detections:
        best_iou, best_idx = 0.0, None
        for idx, gt in enumerate(gt_boxes):
            if idx in matched:
                continue
            iou = compute_iou(det, gt)
            if iou > best_iou:
                best_iou, best_idx = iou, idx
        if best_idx is not None and best_iou >= iou_threshold:
            matched.add(best_idx); tp += 1; matched_ious.append(best_iou)
        else:
            fp += 1
    return dict(tp=tp, fp=fp, fn=len(gt_boxes)-len(matched), matched_ious=matched_ious)

def evaluate_detection_outputs(image_outputs):
    total_images = len(image_outputs)
    gt_count = sum(len(o.get("gt_boxes", [])) for o in image_outputs)
    pred_count = sum(len(o.get("detections", [])) for o in image_outputs)
    base = dict(image_count=total_images, gt_count=int(gt_count), prediction_count=int(pred_count),
                total_window_count=int(sum(o.get("scored_window_count",0) for o in image_outputs)),
                total_candidate_count=int(sum(o.get("candidate_count",0) for o in image_outputs)),
                total_final_detection_count=int(sum(o.get("final_detection_count",0) for o in image_outputs)),
                mean_candidates_per_image=safe_divide(sum(o.get("candidate_count",0) for o in image_outputs), total_images),
                mean_final_detections_per_image=safe_divide(pred_count, total_images),
                total_runtime_seconds=float(sum(o.get("runtime_seconds",0) for o in image_outputs)),
                mean_runtime_seconds_per_image=safe_divide(sum(o.get("runtime_seconds",0) for o in image_outputs), total_images))
    for thr in MATCHING_IOU_THRESHOLDS:
        key = metric_key(thr)
        tp = fp = fn = 0
        ious = []
        for o in image_outputs:
            m = match_detections_to_gt(o.get("detections", []), o.get("gt_boxes", []), thr)
            tp += m["tp"]; fp += m["fp"]; fn += m["fn"]; ious += m["matched_ious"]
        precision = safe_divide(tp, tp+fp); recall = safe_divide(tp, tp+fn); f1 = safe_divide(2*precision*recall, precision+recall)
        base.update({f"tp_iou_{key}": int(tp), f"fp_iou_{key}": int(fp), f"fn_iou_{key}": int(fn),
                     f"detection_precision_iou_{key}": precision, f"detection_recall_iou_{key}": recall,
                     f"detection_f1_iou_{key}": f1, f"false_positives_per_image_iou_{key}": safe_divide(fp, total_images),
                     f"false_negatives_per_image_iou_{key}": safe_divide(fn, total_images),
                     f"mean_matched_iou_iou_{key}": float(np.mean(ious)) if ious else 0.0})
    return base

def image_level_metrics(image_outputs):
    tp=tn=fp=fn=0
    for o in image_outputs:
        gt_has = len(o.get("gt_boxes", [])) > 0
        pred_has = len(o.get("detections", [])) > 0
        if gt_has and pred_has: tp += 1
        elif (not gt_has) and (not pred_has): tn += 1
        elif (not gt_has) and pred_has: fp += 1
        else: fn += 1
    precision = safe_divide(tp, tp+fp); recall = safe_divide(tp, tp+fn); specificity = safe_divide(tn, tn+fp)
    f1 = safe_divide(2*precision*recall, precision+recall)
    return dict(image_tp=tp, image_tn=tn, image_fp=fp, image_fn=fn, image_accuracy=safe_divide(tp+tn,tp+tn+fp+fn),
                image_precision=precision, image_recall=recall, image_specificity=specificity, image_f1=f1,
                false_positive_image_rate=safe_divide(fp, fp+tn), missed_positive_image_rate=safe_divide(fn, fn+tp))

def detections_df_from_outputs(image_outputs, mr):
    rows = []
    mode = "full_external_test_smoke" if RUN_SMOKE_TEST else "full_external_test"
    for o in image_outputs:
        for i, d in enumerate(o.get("detections", []), start=1):
            rows.append(dict(eval_mode=mode, dataset_name=DATASET3_NAME, source_dataset=mr["source_dataset"], source_group=mr["source_group"],
                             algorithm_name=mr["algorithm_name"], model_id=mr["model_id"], model_file_name=mr["model_file_name"],
                             image_id=o["image_id"], detection_index=i, x1=float(d["x1"]), y1=float(d["y1"]),
                             x2=float(d["x2"]), y2=float(d["y2"]), score=float(d.get("score",0)), mean_score=float(d.get("mean_score",d.get("score",0))),
                             cluster_size=int(d.get("cluster_size",1))))
    return pd.DataFrame(rows)

def per_image_df_from_outputs(image_outputs, mr):
    mode = "full_external_test_smoke" if RUN_SMOKE_TEST else "full_external_test"
    return pd.DataFrame([dict(eval_mode=mode, dataset_name=DATASET3_NAME, source_dataset=mr["source_dataset"], source_group=mr["source_group"],
                              algorithm_name=mr["algorithm_name"], model_id=mr["model_id"], model_file_name=mr["model_file_name"],
                              image_id=o["image_id"], gt_count=len(o.get("gt_boxes",[])), prediction_count=len(o.get("detections",[])),
                              scored_window_count=o.get("scored_window_count",0), candidate_count=o.get("candidate_count",0),
                              final_detection_count=o.get("final_detection_count",0), stride_px=o.get("stride_px",np.nan),
                              runtime_seconds=o.get("runtime_seconds",0.0)) for o in image_outputs])

def compute_ap_for_iou(detections_df, image_records_df, iou_threshold):
    gt_by_image = {str(r["image_id"]): list(r["gt_boxes"]) for _, r in image_records_df.iterrows()}
    total_gt = sum(len(v) for v in gt_by_image.values())
    if total_gt == 0 or detections_df.empty:
        return 0.0
    dets = detections_df.sort_values("score", ascending=False).to_dict("records")
    matched = {img_id: set() for img_id in gt_by_image}
    tp_flags, fp_flags = [], []
    for det in dets:
        img_id = str(det["image_id"])
        best_iou, best_idx = 0.0, None
        for idx, gt in enumerate(gt_by_image.get(img_id, [])):
            if idx in matched[img_id]:
                continue
            iou = compute_iou(det, gt)
            if iou > best_iou:
                best_iou, best_idx = iou, idx
        if best_idx is not None and best_iou >= iou_threshold:
            matched[img_id].add(best_idx); tp_flags.append(1); fp_flags.append(0)
        else:
            tp_flags.append(0); fp_flags.append(1)
    tp_cum, fp_cum = np.cumsum(tp_flags), np.cumsum(fp_flags)
    rec = tp_cum / max(total_gt, 1); prec = tp_cum / np.maximum(tp_cum+fp_cum, 1)
    mrec = np.concatenate(([0.0], rec, [1.0])); mpre = np.concatenate(([0.0], prec, [0.0]))
    for i in range(len(mpre)-1, 0, -1):
        mpre[i-1] = max(mpre[i-1], mpre[i])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx+1] - mrec[idx]) * mpre[idx+1]))

def compute_ap_map(detections_df, image_records_df):
    row = {}
    vals_50_95 = []
    for thr in sorted(set(AP_IOU_THRESHOLDS + MAP_50_95_THRESHOLDS)):
        ap = compute_ap_for_iou(detections_df, image_records_df, thr)
        row[f"ap_iou_{metric_key(thr)}"] = ap
        if 0.50 <= thr <= 0.95:
            vals_50_95.append(ap)
    row["mAP50"] = row.get("ap_iou_0_50", 0.0)
    row["mAP50_95"] = float(np.mean(vals_50_95)) if vals_50_95 else 0.0
    return row

def partial_paths(model_id):
    mode = "full_external_test_smoke" if RUN_SMOKE_TEST else "full_external_test"
    prefix = f"{mode}__{DATASET3_NAME}__{model_id}"
    return {k: PARTIAL_DIR / f"{prefix}__{name}.csv" for k, name in {
        "summary": "detection_summary", "image_level": "image_level_presence",
        "per_image": "per_image_records", "detections": "final_detections", "ap_map": "ap_map"}.items()}

def evaluate_external_model(mr):
    paths = partial_paths(mr["model_id"])
    if paths["summary"].exists() and not OVERWRITE_PARTIAL_DETECTION_RESULTS:
        return (pd.read_csv(paths["summary"]),
                pd.read_csv(paths["image_level"]) if paths["image_level"].exists() else pd.DataFrame(),
                pd.read_csv(paths["per_image"]) if paths["per_image"].exists() else pd.DataFrame(),
                pd.read_csv(paths["detections"]) if paths["detections"].exists() else pd.DataFrame(),
                pd.read_csv(paths["ap_map"]) if paths["ap_map"].exists() else pd.DataFrame())

    model = load_model(mr["model_path_resolved"])
    outs = []
    print(f"\\n[RUN] {mr['model_id']} images={len(dataset3_eval_records_df)}")
    for i, (_, ir) in enumerate(dataset3_eval_records_df.iterrows(), start=1):
        try:
            outs.append(run_detection_for_image(model, ir, mr))
        except Exception as exc:
            warnings.warn(f"Detection failed model={mr['model_id']} image={ir.get('image_id')}: {exc}")
            outs.append(dict(image_id=ir.get("image_id"), gt_boxes=ir.get("gt_boxes", []), detections=[],
                             scored_window_count=0, candidate_count=0, final_detection_count=0, stride_px=np.nan,
                             runtime_seconds=0.0, error=repr(exc)))
        if i % 25 == 0 or i == len(dataset3_eval_records_df):
            print(f"  processed {i}/{len(dataset3_eval_records_df)}")

    mode = "full_external_test_smoke" if RUN_SMOKE_TEST else "full_external_test"
    summary = evaluate_detection_outputs(outs)
    summary.update(dict(eval_mode=mode, dataset_name=DATASET3_NAME, source_dataset=mr["source_dataset"], source_group=mr["source_group"],
                        algorithm_name=mr["algorithm_name"], model_id=mr["model_id"], model_file_name=mr["model_file_name"],
                        patch_set=mr["patch_set"], feature_set=mr["feature_set"], threshold_value=float(mr["threshold_value"]),
                        stride_divisor=int(mr["stride_divisor"]), cluster_iou_threshold=float(mr["cluster_iou_threshold"]),
                        weighted_box_size_factor=float(mr["weighted_box_size_factor"]), status="OK", run_smoke_test=RUN_SMOKE_TEST))
    sdf = pd.DataFrame([summary])

    il = image_level_metrics(outs)
    il.update(dict(eval_mode=mode, dataset_name=DATASET3_NAME, source_dataset=mr["source_dataset"], source_group=mr["source_group"],
                   algorithm_name=mr["algorithm_name"], model_id=mr["model_id"], model_file_name=mr["model_file_name"], run_smoke_test=RUN_SMOKE_TEST))
    ildf = pd.DataFrame([il])

    pidf = per_image_df_from_outputs(outs, mr)
    ddf = detections_df_from_outputs(outs, mr)
    apdf = pd.DataFrame([{**compute_ap_map(ddf, dataset3_eval_records_df), "eval_mode": mode, "dataset_name": DATASET3_NAME,
                          "source_dataset": mr["source_dataset"], "source_group": mr["source_group"], "algorithm_name": mr["algorithm_name"],
                          "model_id": mr["model_id"], "model_file_name": mr["model_file_name"], "run_smoke_test": RUN_SMOKE_TEST}]) if COMPUTE_AP_MAP else pd.DataFrame()

    save_csv(sdf, paths["summary"], "Partial external detection summary", overwrite=True)
    save_csv(ildf, paths["image_level"], "Partial external image-level metrics", overwrite=True)
    save_csv(pidf, paths["per_image"], "Partial external per-image records", overwrite=True)
    save_csv(ddf, paths["detections"], "Partial external final detections", overwrite=True)
    save_csv(apdf, paths["ap_map"], "Partial external AP/mAP metrics", overwrite=True)
    return sdf, ildf, pidf, ddf, apdf

all_summary, all_image_level, all_per_image, all_detections, all_ap = [], [], [], [], []
if RUN_EXTERNAL_DETECTION:
    for _, mr in finalists_df.iterrows():
        out = evaluate_external_model(mr)
        all_summary.append(out[0]); all_image_level.append(out[1]); all_per_image.append(out[2]); all_detections.append(out[3]); all_ap.append(out[4])
        gc.collect()

external_detection_results_df = pd.concat(all_summary, ignore_index=True) if all_summary else pd.DataFrame()
external_image_level_df = pd.concat(all_image_level, ignore_index=True) if all_image_level else pd.DataFrame()
external_per_image_df = pd.concat(all_per_image, ignore_index=True) if all_per_image else pd.DataFrame()
external_final_detections_df = pd.concat(all_detections, ignore_index=True) if all_detections else pd.DataFrame()
external_ap_map_df = pd.concat(all_ap, ignore_index=True) if all_ap else pd.DataFrame()

rank020 = external_detection_results_df.sort_values(["detection_f1_iou_0_20","detection_recall_iou_0_20","detection_precision_iou_0_20"], ascending=[False,False,False]) if not external_detection_results_df.empty else external_detection_results_df
rank030 = external_detection_results_df.sort_values(["detection_f1_iou_0_30","detection_recall_iou_0_30","detection_precision_iou_0_30"], ascending=[False,False,False]) if not external_detection_results_df.empty else external_detection_results_df

save_csv(external_detection_results_df, TABLES_DIR / "external_detection_results.csv", "External detection results")
save_csv(rank020, TABLES_DIR / "ranked_external_detection_results_by_iou_0_20.csv", "External detection ranked by F1@0.20")
save_csv(rank030, TABLES_DIR / "ranked_external_detection_results_by_iou_0_30.csv", "External detection ranked by F1@0.30")
save_csv(external_image_level_df, TABLES_DIR / "external_image_level_presence_results.csv", "External image-level presence results")
save_csv(external_ap_map_df, TABLES_DIR / "external_ap_map_results.csv", "External AP/mAP results")
save_csv(external_per_image_df, TABLES_DIR / "external_per_image_detection_records.csv", "External per-image records")
save_csv(external_final_detections_df, TABLES_DIR / "external_final_detections.csv", "External final detections")

log_dataframe("Ranked external detection by F1@0.20", rank020)
log_dataframe("External image-level results", external_image_level_df)
log_dataframe("External AP/mAP results", external_ap_map_df)


## 9. Final Durum

Bu bölümde dış doğrulama özet tabloları, notebook durumu ve ana çıktı listesi kaydedilir.


In [ ]:
group_rows = []
if not external_detection_results_df.empty:
    for source_group, group_df in external_detection_results_df.groupby("source_group"):
        group_rows.append({
            "source_group": source_group,
            "model_count": len(group_df),
            "mean_detection_f1_iou_0_20": float(group_df["detection_f1_iou_0_20"].mean()),
            "max_detection_f1_iou_0_20": float(group_df["detection_f1_iou_0_20"].max()),
            "mean_detection_f1_iou_0_30": float(group_df["detection_f1_iou_0_30"].mean()),
            "max_detection_f1_iou_0_30": float(group_df["detection_f1_iou_0_30"].max()),
            "mean_false_positives_per_image_iou_0_20": float(group_df["false_positives_per_image_iou_0_20"].mean()),
        })

source_dataset_group_summary_df = pd.DataFrame(group_rows)
save_csv(source_dataset_group_summary_df, TABLES_DIR / "source_dataset_group_summary.csv", "Source-dataset group transfer summary")
log_dataframe("Source-dataset group summary", source_dataset_group_summary_df)

completed_detection_models = int((external_detection_results_df.get("status", pd.Series(dtype=str)) == "OK").sum()) if not external_detection_results_df.empty else 0
failed_detection_models = int((external_detection_results_df.get("status", pd.Series(dtype=str)) == "FAILED").sum()) if not external_detection_results_df.empty else 0

status_df = pd.DataFrame([{
    "stage_name": STAGE_NAME,
    "notebook_name": NOTEBOOK_NAME,
    "status": "SMOKE_TEST_READY_FOR_REVIEW" if RUN_SMOKE_TEST else ("READY_FOR_REVIEW" if failed_detection_models == 0 else "COMPLETED_WITH_FAILURES"),
    "started_at": NOTEBOOK_STARTED_AT.isoformat(timespec="seconds"),
    "finished_at": datetime.now().isoformat(timespec="seconds"),
    "run_smoke_test": RUN_SMOKE_TEST,
    "run_exploration_audit": RUN_EXPLORATION_AUDIT,
    "run_external_classification_metadata": RUN_EXTERNAL_CLASSIFICATION_METADATA,
    "run_external_classification_features": RUN_EXTERNAL_CLASSIFICATION_FEATURES,
    "run_external_classification_evaluation": RUN_EXTERNAL_CLASSIFICATION_EVALUATION,
    "run_external_detection": RUN_EXTERNAL_DETECTION,
    "compute_ap_map": COMPUTE_AP_MAP,
    "expected_finalist_count": 5,
    "completed_detection_models": completed_detection_models,
    "failed_detection_models": failed_detection_models,
    "notes": "Stage 11 external validation. No training, tuning, PCA refit, scaler refit, or Dataset3-based parameter selection.",
}])
save_csv(status_df, TABLES_DIR / "notebook_status.csv", "Stage 11 notebook status")

output_summary_df = pd.DataFrame([
    {"output_name": "dataset3_external_test_inventory.csv", "path": relative_path(TABLES_DIR / "dataset3_external_test_inventory.csv")},
    {"output_name": "dataset3_exploration_audit.csv", "path": relative_path(TABLES_DIR / "dataset3_exploration_audit.csv")},
    {"output_name": "external_classification_results.csv", "path": relative_path(TABLES_DIR / "external_classification_results.csv")},
    {"output_name": "external_detection_results.csv", "path": relative_path(TABLES_DIR / "external_detection_results.csv")},
    {"output_name": "ranked_external_detection_results_by_iou_0_20.csv", "path": relative_path(TABLES_DIR / "ranked_external_detection_results_by_iou_0_20.csv")},
    {"output_name": "ranked_external_detection_results_by_iou_0_30.csv", "path": relative_path(TABLES_DIR / "ranked_external_detection_results_by_iou_0_30.csv")},
    {"output_name": "external_image_level_presence_results.csv", "path": relative_path(TABLES_DIR / "external_image_level_presence_results.csv")},
    {"output_name": "external_ap_map_results.csv", "path": relative_path(TABLES_DIR / "external_ap_map_results.csv")},
    {"output_name": "source_dataset_group_summary.csv", "path": relative_path(TABLES_DIR / "source_dataset_group_summary.csv")},
])
save_csv(output_summary_df, TABLES_DIR / "stage11_output_summary.csv", "Stage 11 output summary")

display(status_df)
display(output_summary_df)
